# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a demonstration of how to load and explore a Croissant dataset using the `mlcroissant` library. It guides you through loading metadata, discovering record sets and fields using their `@id`s, extracting data by referencing IDs, conducting basic preprocessing, and visualizing the results.

### Dataset Source
The dataset is described with a Croissant schema, accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and initialize access to its record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata summary
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Version: {metadata.version}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review the available record sets, their `@id`s, and the fields and columns present in each record set. All entities are referenced by their `@id`.

In [ ]:
# List available record sets and their field @id's
if hasattr(metadata, 'record_sets'):
    print('Available record sets:')
    for rs in metadata.record_sets:
        print(f"  - Record Set Name: {rs.name}")
        print(f"    @id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            print(f"    Field @ids:")
            for fld in rs.fields:
                print(f"      - {fld.id} ({fld.name})")
        if hasattr(rs, 'columns') and rs.columns:
            print(f"    Column @ids:")
            for col in rs.columns:
                print(f"      - {col.id} ({col.name})")
else:
    print('No record sets found in metadata.')

## 3. Data Extraction

Load data from a selected record set into a DataFrame for analysis. Use the record set `@id` and its field/column `@id`s as listed above. All references are made using `@id`s for clarity.

In [ ]:
# Discover all available record set @id's
record_sets = []
if hasattr(metadata, 'record_sets'):
    record_sets = [rs.id for rs in metadata.record_sets]

# Choose one record set for main analysis (the first one, for example)
if record_sets:
    selected_record_set_id = record_sets[0]
    print(f"Using record set @id: {selected_record_set_id}")
else:
    raise ValueError('No record sets found in the dataset.')

# Extract data from each record set
dataframes = {}
for record_set in record_sets:
    try:
        records = list(dataset.records(record_set=record_set))
        dataframes[record_set] = pd.DataFrame(records)
        print(f"Loaded records for record set: {record_set}, shape: {dataframes[record_set].shape}")
    except Exception as e:
        print(f"Could not load data for record set {record_set}: {e}")

# Show columns of the main record set
print(f"Field/column @ids in DataFrame for '{selected_record_set_id}':")
print(dataframes[selected_record_set_id].columns.tolist())
dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply basic data processing steps such as filtering on a numeric field, normalization, and grouping. All fields/columns should be referenced using their `@id`s as determined in the previous section.

In [ ]:
# Determine candidate numeric and group fields by inspecting DataFrame columns
df = dataframes[selected_record_set_id]
print('Columns:', df.columns.tolist())

# Select the first numeric-looking column for illustrative filtering
possible_numeric_fields = [col for col in df.columns if (('log' in col.lower()) or ('value' in col.lower()) or df[col].dtype in [int, float])]
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
    print(f"Using numeric field @id: {numeric_field_id}")
else:
    raise ValueError('No numeric-like field found in data. Please inspect columns and select manually.')

# Choose a threshold for filtering
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0

# Filter and normalize
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean):")
print(filtered_df.head())

# Normalize the numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized '{numeric_field_id}':")
print(filtered_df[[numeric_field_id, norm_col]].head())

# Try grouping by a likely categorical/grouping field
group_fields = [col for col in df.columns if ('gender' in col.lower()) or ('ward' in col.lower()) or ('county' in col.lower())]
if group_fields:
    group_field_id = group_fields[0]
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean')
        print(f"Grouped data by {group_field_id} and calculated mean {numeric_field_id}:")
        print(grouped_df.head())
else:
    print('No suitable group field (@id) found for grouping example.')

## 5. Visualization

Visualize distributions or relationships between fields in the dataset, referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field after filtering
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id], kde=True)
plt.title(f"Distribution of '{numeric_field_id}' (filtered)")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# Optional: boxplot by group if group_field_id exists
if 'group_field_id' in locals() and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"'{numeric_field_id}' by '{group_field_id}' (filtered)")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we used `mlcroissant` to:

- Load metadata and review available record sets and their `@id`s
- Extract records using record set and field `@id`s
- Filter, normalize, and group data referencing fields by `@id`
- Visualize results via histograms and boxplots

By referencing all dataset entities with their proper `@id`s, we maintain clarity and standards compliance, and enable reproducible analysis with Croissant-compliant datasets.